# 3 - Fixed Inner 80/20 Train-Validation Splits - All Families

For each ligand family and each outer fold:

1. the selected outer fold remains the **untouched outer test set**;
2. the remaining four folds form the **outer-training set**;
3. the outer-training set is split stratifiably into:
   - **80% inner training**
   - **20% inner validation**

The same procedure is used for:

- Enzyme
- GPCR
- Ion Channel
- Nuclear Receptor



In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
SPLIT_DIR = PROJECT_ROOT / "data" / "split"
STATS_DIR = PROJECT_ROOT / "Stats"

STATS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Split folder :", SPLIT_DIR)
print("Stats folder :", STATS_DIR)

assert SPLIT_DIR.exists(), f"Split folder not found: {SPLIT_DIR}"

Project root : c:\Users\riskf\OneDrive\A-DTI2026
Split folder : c:\Users\riskf\OneDrive\A-DTI2026\data\split
Stats folder : c:\Users\riskf\OneDrive\A-DTI2026\Stats


## 1. Inner split configuration

```python
OUTER_SPLIT_SEED = 13571113
```



In [2]:
OUTER_SPLIT_SEED = 13571113

INNER_TRAIN_SIZE = 0.80
INNER_VALIDATION_SIZE = 0.20

# ---------------------------------------------------------------------
# REQUIRED: choose one fixed integer seed for the inner split.
# Do not change it after the assignments have been generated.
# ---------------------------------------------------------------------
INNER_SPLIT_SEED = 555487

assert isinstance(INNER_SPLIT_SEED, int), (
    "Set INNER_SPLIT_SEED to the fixed integer seed selected for "
    "the inner 80/20 train-validation split."
)

FAMILIES = {
    "enzyme": "Enzyme",
    "gpcr": "GPCR",
    "ion_channel": "Ion Channel",
    "nuclear_receptor": "Nuclear Receptor",
}

N_OUTER_FOLDS = 5

print("Outer split seed      :", OUTER_SPLIT_SEED)
print("Inner split seed      :", INNER_SPLIT_SEED)
print("Inner training share  :", INNER_TRAIN_SIZE)
print("Inner validation share:", INNER_VALIDATION_SIZE)

Outer split seed      : 13571113
Inner split seed      : 555487
Inner training share  : 0.8
Inner validation share: 0.2


## 2. Load the permanent outer-fold assignments



In [3]:
OUTER_DATA = {}

required_columns = {
    "pair_id",
    "family",
    "compound_id",
    "protein_id",
    "compound_index",
    "protein_index",
    "y",
    "outer_fold",
}

for family, display_name in FAMILIES.items():

    path = SPLIT_DIR / family / "outer_folds.csv"

    assert path.exists(), (
        f"{display_name}: missing outer-fold file: {path}"
    )

    df = pd.read_csv(path)

    missing = required_columns - set(df.columns)

    assert not missing, (
        f"{display_name}: missing required columns: {sorted(missing)}"
    )

    assert df["pair_id"].is_unique, (
        f"{display_name}: pair_id is not unique in outer_folds.csv"
    )

    assert set(df["outer_fold"].unique()) == {1, 2, 3, 4, 5}, (
        f"{display_name}: outer-fold labels are not exactly 1-5."
    )

    assert set(df["y"].unique()).issubset({0, 1}), (
        f"{display_name}: invalid labels in y."
    )

    OUTER_DATA[family] = df

    print(
        f"{display_name:<18} "
        f"pairs={len(df):,} | "
        f"positive={(df['y'] == 1).sum():,} | "
        f"non-interaction={(df['y'] == 0).sum():,}"
    )

Enzyme             pairs=295,480 | positive=2,926 | non-interaction=292,554
GPCR               pairs=21,185 | positive=635 | non-interaction=20,550
Ion Channel        pairs=42,840 | positive=1,476 | non-interaction=41,364
Nuclear Receptor   pairs=1,404 | positive=90 | non-interaction=1,314


## 3. Create fixed inner train-validation assignments

For each outer fold:

```text
outer_fold == k       -> test
outer_fold != k       -> outer-training pool
```

The outer-training pool is then split with stratification on `y`:

```text
80% -> train
20% -> validation
```

The outer test fold is never passed to `train_test_split`.

In [4]:
INNER_ASSIGNMENTS = {}

for family, display_name in FAMILIES.items():

    df = OUTER_DATA[family]

    family_assignments = {}

    for outer_fold in range(1, N_OUTER_FOLDS + 1):

        test_df = df[df["outer_fold"] == outer_fold].copy()
        outer_train_df = df[df["outer_fold"] != outer_fold].copy()

        train_df, validation_df = train_test_split(
            outer_train_df,
            test_size=INNER_VALIDATION_SIZE,
            random_state=INNER_SPLIT_SEED,
            shuffle=True,
            stratify=outer_train_df["y"],
        )

        # Preserve original order after splitting.
        train_df = train_df.sort_index()
        validation_df = validation_df.sort_index()
        test_df = test_df.sort_index()

        train_df = train_df.copy()
        validation_df = validation_df.copy()
        test_df = test_df.copy()

        train_df["role"] = "train"
        validation_df["role"] = "validation"
        test_df["role"] = "test"

        assignment = pd.concat(
            [train_df, validation_df, test_df],
            axis=0,
        ).sort_index()

        # One role per pair within this outer fold.
        assert len(assignment) == len(df)
        assert assignment["pair_id"].is_unique

        assignment["evaluation_outer_fold"] = outer_fold

        family_assignments[outer_fold] = assignment

        print(
            f"{display_name:<18} fold={outer_fold} | "
            f"train={len(train_df):,} | "
            f"validation={len(validation_df):,} | "
            f"test={len(test_df):,}"
        )

    INNER_ASSIGNMENTS[family] = family_assignments

Enzyme             fold=1 | train=189,107 | validation=47,277 | test=59,096
Enzyme             fold=2 | train=189,107 | validation=47,277 | test=59,096
Enzyme             fold=3 | train=189,107 | validation=47,277 | test=59,096
Enzyme             fold=4 | train=189,107 | validation=47,277 | test=59,096
Enzyme             fold=5 | train=189,107 | validation=47,277 | test=59,096
GPCR               fold=1 | train=13,558 | validation=3,390 | test=4,237
GPCR               fold=2 | train=13,558 | validation=3,390 | test=4,237
GPCR               fold=3 | train=13,558 | validation=3,390 | test=4,237
GPCR               fold=4 | train=13,558 | validation=3,390 | test=4,237
GPCR               fold=5 | train=13,558 | validation=3,390 | test=4,237
Ion Channel        fold=1 | train=27,417 | validation=6,855 | test=8,568
Ion Channel        fold=2 | train=27,417 | validation=6,855 | test=8,568
Ion Channel        fold=3 | train=27,417 | validation=6,855 | test=8,568
Ion Channel        fold=4 | train=27

## 4. Role-count and class-distribution checks

For every family and outer fold, verify:

- train, validation, and test are non-empty;
- each role contains both classes;
- the three roles sum to the full original dataset;
- the test role is exactly the original outer test fold;
- the train and validation roles jointly equal the original outer-training pool.

In [5]:
ROLE_STATS_ROWS = []
FOLD_SUMMARY_ROWS = []

for family, display_name in FAMILIES.items():

    original = OUTER_DATA[family]

    for outer_fold in range(1, N_OUTER_FOLDS + 1):

        assignment = INNER_ASSIGNMENTS[family][outer_fold]

        train_df = assignment[assignment["role"] == "train"]
        validation_df = assignment[assignment["role"] == "validation"]
        test_df = assignment[assignment["role"] == "test"]

        # Role counts
        assert len(train_df) > 0
        assert len(validation_df) > 0
        assert len(test_df) > 0

        # Both classes must be present in every role.
        for role_name, role_df in [
            ("train", train_df),
            ("validation", validation_df),
            ("test", test_df),
        ]:
            assert set(role_df["y"].unique()) == {0, 1}, (
                f"{display_name}, outer fold {outer_fold}, {role_name}: "
                "both classes are not present."
            )

        # Complete partition.
        assert (
            len(train_df) + len(validation_df) + len(test_df)
            == len(original)
        )

        # Test must exactly equal the original outer fold.
        expected_test_ids = set(
            original.loc[
                original["outer_fold"] == outer_fold,
                "pair_id"
            ]
        )
        actual_test_ids = set(test_df["pair_id"])

        assert actual_test_ids == expected_test_ids, (
            f"{display_name}, outer fold {outer_fold}: "
            "test IDs do not match the permanent outer fold."
        )

        # Train + validation must equal the outer-training pool.
        expected_outer_train_ids = set(
            original.loc[
                original["outer_fold"] != outer_fold,
                "pair_id"
            ]
        )
        actual_outer_train_ids = (
            set(train_df["pair_id"])
            | set(validation_df["pair_id"])
        )

        assert actual_outer_train_ids == expected_outer_train_ids, (
            f"{display_name}, outer fold {outer_fold}: "
            "train + validation does not equal the outer-training pool."
        )

        for role_name, role_df in [
            ("train", train_df),
            ("validation", validation_df),
            ("test", test_df),
        ]:

            n_total = len(role_df)
            n_positive = int((role_df["y"] == 1).sum())
            n_non_interaction = int((role_df["y"] == 0).sum())

            ROLE_STATS_ROWS.append({
                "family": display_name,
                "outer_fold": outer_fold,
                "role": role_name,
                "n_total": n_total,
                "n_positive": n_positive,
                "n_non_interaction": n_non_interaction,
                "positive_rate": n_positive / n_total,
                "non_interaction_rate": n_non_interaction / n_total,
            })

        FOLD_SUMMARY_ROWS.append({
            "family": display_name,
            "outer_fold": outer_fold,
            "n_original": len(original),
            "n_train": len(train_df),
            "n_validation": len(validation_df),
            "n_test": len(test_df),
            "train_share_total_dataset": len(train_df) / len(original),
            "validation_share_total_dataset": len(validation_df) / len(original),
            "test_share_total_dataset": len(test_df) / len(original),
            "train_share_outer_training": (
                len(train_df) / (len(train_df) + len(validation_df))
            ),
            "validation_share_outer_training": (
                len(validation_df) / (len(train_df) + len(validation_df))
            ),
        })

        print(f"PASS - {display_name}, outer fold {outer_fold}")

PASS - Enzyme, outer fold 1
PASS - Enzyme, outer fold 2
PASS - Enzyme, outer fold 3
PASS - Enzyme, outer fold 4
PASS - Enzyme, outer fold 5
PASS - GPCR, outer fold 1
PASS - GPCR, outer fold 2
PASS - GPCR, outer fold 3
PASS - GPCR, outer fold 4
PASS - GPCR, outer fold 5
PASS - Ion Channel, outer fold 1
PASS - Ion Channel, outer fold 2
PASS - Ion Channel, outer fold 3
PASS - Ion Channel, outer fold 4
PASS - Ion Channel, outer fold 5
PASS - Nuclear Receptor, outer fold 1
PASS - Nuclear Receptor, outer fold 2
PASS - Nuclear Receptor, outer fold 3
PASS - Nuclear Receptor, outer fold 4
PASS - Nuclear Receptor, outer fold 5


In [6]:
ROLE_STATS = pd.DataFrame(ROLE_STATS_ROWS)
FOLD_SUMMARY = pd.DataFrame(FOLD_SUMMARY_ROWS)

display(ROLE_STATS)
display(FOLD_SUMMARY)

,family,outer_fold,role,n_total,n_positive,n_non_interaction,positive_rate,non_interaction_rate
0,Enzyme,1,train,189107,1872,187235,0.009899,0.990101
1,Enzyme,1,validation,47277,468,46809,0.009899,0.990101
2,Enzyme,1,test,59096,586,58510,0.009916,0.990084
3,Enzyme,2,train,189107,1873,187234,0.009904,0.990096
4,Enzyme,2,validation,47277,468,46809,0.009899,0.990101
5,Enzyme,2,test,59096,585,58511,0.009899,0.990101
6,Enzyme,3,train,189107,1873,187234,0.009904,0.990096
7,Enzyme,3,validation,47277,468,46809,0.009899,0.990101
8,Enzyme,3,test,59096,585,58511,0.009899,0.990101
9,Enzyme,4,train,189107,1873,187234,0.009904,0.990096


,family,outer_fold,n_original,n_train,n_validation,n_test,train_share_total_dataset,validation_share_total_dataset,test_share_total_dataset,train_share_outer_training,validation_share_outer_training
0,Enzyme,1,295480,189107,47277,59096,0.639999,0.160001,0.200000,0.799999,0.200001
1,Enzyme,2,295480,189107,47277,59096,0.639999,0.160001,0.200000,0.799999,0.200001
2,Enzyme,3,295480,189107,47277,59096,0.639999,0.160001,0.200000,0.799999,0.200001
3,Enzyme,4,295480,189107,47277,59096,0.639999,0.160001,0.200000,0.799999,0.200001
4,Enzyme,5,295480,189107,47277,59096,0.639999,0.160001,0.200000,0.799999,0.200001
5,GPCR,1,21185,13558,3390,4237,0.639981,0.160019,0.200000,0.799976,0.200024
6,GPCR,2,21185,13558,3390,4237,0.639981,0.160019,0.200000,0.799976,0.200024
7,GPCR,3,21185,13558,3390,4237,0.639981,0.160019,0.200000,0.799976,0.200024
8,GPCR,4,21185,13558,3390,4237,0.639981,0.160019,0.200000,0.799976,0.200024
9,GPCR,5,21185,13558,3390,4237,0.639981,0.160019,0.200000,0.799976,0.200024


## 5. Pairwise overlap checks



In [7]:
OVERLAP_ROWS = []

for family, display_name in FAMILIES.items():

    for outer_fold in range(1, N_OUTER_FOLDS + 1):

        assignment = INNER_ASSIGNMENTS[family][outer_fold]

        train_ids = set(
            assignment.loc[assignment["role"] == "train", "pair_id"]
        )
        validation_ids = set(
            assignment.loc[assignment["role"] == "validation", "pair_id"]
        )
        test_ids = set(
            assignment.loc[assignment["role"] == "test", "pair_id"]
        )

        train_validation_overlap = len(train_ids & validation_ids)
        train_test_overlap = len(train_ids & test_ids)
        validation_test_overlap = len(validation_ids & test_ids)

        OVERLAP_ROWS.append({
            "family": display_name,
            "outer_fold": outer_fold,
            "train_validation_overlap": train_validation_overlap,
            "train_test_overlap": train_test_overlap,
            "validation_test_overlap": validation_test_overlap,
            "pass": (
                train_validation_overlap == 0
                and train_test_overlap == 0
                and validation_test_overlap == 0
            ),
        })

OVERLAP_CHECK = pd.DataFrame(OVERLAP_ROWS)

assert OVERLAP_CHECK["pass"].all(), (
    "At least one train-validation-test overlap was detected."
)

display(OVERLAP_CHECK)

,family,outer_fold,train_validation_overlap,train_test_overlap,validation_test_overlap,pass
0,Enzyme,1,0,0,0,True
1,Enzyme,2,0,0,0,True
2,Enzyme,3,0,0,0,True
3,Enzyme,4,0,0,0,True
4,Enzyme,5,0,0,0,True
5,GPCR,1,0,0,0,True
6,GPCR,2,0,0,0,True
7,GPCR,3,0,0,0,True
8,GPCR,4,0,0,0,True
9,GPCR,5,0,0,0,True


## 6. Stratification-rate checks



In [8]:
RATE_CHECK_ROWS = []

for family, display_name in FAMILIES.items():

    original = OUTER_DATA[family]
    overall_rate = original["y"].mean()

    for outer_fold in range(1, N_OUTER_FOLDS + 1):

        assignment = INNER_ASSIGNMENTS[family][outer_fold]

        train_df = assignment[assignment["role"] == "train"]
        validation_df = assignment[assignment["role"] == "validation"]
        test_df = assignment[assignment["role"] == "test"]

        outer_train_df = assignment[
            assignment["role"].isin(["train", "validation"])
        ]

        for role_name, role_df in [
            ("overall", original),
            ("outer_training", outer_train_df),
            ("train", train_df),
            ("validation", validation_df),
            ("test", test_df),
        ]:

            role_rate = role_df["y"].mean()

            RATE_CHECK_ROWS.append({
                "family": display_name,
                "outer_fold": outer_fold,
                "dataset_role": role_name,
                "positive_rate": role_rate,
                "overall_positive_rate": overall_rate,
                "absolute_difference_from_overall": abs(
                    role_rate - overall_rate
                ),
            })

INNER_RATE_CHECK = pd.DataFrame(RATE_CHECK_ROWS)

display(INNER_RATE_CHECK)

,family,outer_fold,dataset_role,positive_rate,overall_positive_rate,absolute_difference_from_overall
0,Enzyme,1,overall,0.009903,0.009903,0.000000
1,Enzyme,1,outer_training,0.009899,0.009903,0.000003
2,Enzyme,1,train,0.009899,0.009903,0.000003
3,Enzyme,1,validation,0.009899,0.009903,0.000003
4,Enzyme,1,test,0.009916,0.009903,0.000014
...,...,...,...,...,...,...
95,Nuclear Receptor,5,overall,0.064103,0.064103,0.000000
96,Nuclear Receptor,5,outer_training,0.064057,0.064103,0.000046
97,Nuclear Receptor,5,train,0.064516,0.064103,0.000414
98,Nuclear Receptor,5,validation,0.062222,0.064103,0.001880


## 7. Save permanent role assignments



In [9]:
for family, display_name in FAMILIES.items():

    family_dir = SPLIT_DIR / family
    family_dir.mkdir(parents=True, exist_ok=True)

    for outer_fold in range(1, N_OUTER_FOLDS + 1):

        output_file = (
            family_dir / f"outer_fold_{outer_fold}_roles.csv"
        )

        INNER_ASSIGNMENTS[family][outer_fold].to_csv(
            output_file,
            index=False,
        )

        print(f"Saved - {display_name}, fold {outer_fold}: {output_file}")

Saved - Enzyme, fold 1: c:\Users\riskf\OneDrive\A-DTI2026\data\split\enzyme\outer_fold_1_roles.csv
Saved - Enzyme, fold 2: c:\Users\riskf\OneDrive\A-DTI2026\data\split\enzyme\outer_fold_2_roles.csv
Saved - Enzyme, fold 3: c:\Users\riskf\OneDrive\A-DTI2026\data\split\enzyme\outer_fold_3_roles.csv
Saved - Enzyme, fold 4: c:\Users\riskf\OneDrive\A-DTI2026\data\split\enzyme\outer_fold_4_roles.csv
Saved - Enzyme, fold 5: c:\Users\riskf\OneDrive\A-DTI2026\data\split\enzyme\outer_fold_5_roles.csv
Saved - GPCR, fold 1: c:\Users\riskf\OneDrive\A-DTI2026\data\split\gpcr\outer_fold_1_roles.csv
Saved - GPCR, fold 2: c:\Users\riskf\OneDrive\A-DTI2026\data\split\gpcr\outer_fold_2_roles.csv
Saved - GPCR, fold 3: c:\Users\riskf\OneDrive\A-DTI2026\data\split\gpcr\outer_fold_3_roles.csv
Saved - GPCR, fold 4: c:\Users\riskf\OneDrive\A-DTI2026\data\split\gpcr\outer_fold_4_roles.csv
Saved - GPCR, fold 5: c:\Users\riskf\OneDrive\A-DTI2026\data\split\gpcr\outer_fold_5_roles.csv
Saved - Ion Channel, fold 1: c

## 8. Save every statistical/check DataFrame under `Stats/`



In [10]:
# ---------------------------------------------------------------------
# Individual Excel files
# ---------------------------------------------------------------------

ROLE_STATS.to_excel(
    STATS_DIR / "inner_split_role_statistics.xlsx",
    index=False,
    sheet_name="Role Statistics",
)

FOLD_SUMMARY.to_excel(
    STATS_DIR / "inner_split_fold_summary.xlsx",
    index=False,
    sheet_name="Fold Summary",
)

OVERLAP_CHECK.to_excel(
    STATS_DIR / "inner_split_overlap_check.xlsx",
    index=False,
    sheet_name="Overlap Check",
)

INNER_RATE_CHECK.to_excel(
    STATS_DIR / "inner_split_rate_check.xlsx",
    index=False,
    sheet_name="Rate Check",
)

print("Saved individual statistics/check files to Stats/.")

Saved individual statistics/check files to Stats/.


## 9. Reproducibility check



In [11]:
REPRODUCIBILITY_ROWS = []

for family, display_name in FAMILIES.items():

    original = OUTER_DATA[family]

    for outer_fold in range(1, N_OUTER_FOLDS + 1):

        saved = pd.read_csv(
            SPLIT_DIR
            / family
            / f"outer_fold_{outer_fold}_roles.csv"
        )

        outer_train_df = original[
            original["outer_fold"] != outer_fold
        ].copy()

        test_df = original[
            original["outer_fold"] == outer_fold
        ].copy()

        train_df, validation_df = train_test_split(
            outer_train_df,
            test_size=INNER_VALIDATION_SIZE,
            random_state=INNER_SPLIT_SEED,
            shuffle=True,
            stratify=outer_train_df["y"],
        )

        regenerated_roles = {}

        for pair_id in train_df["pair_id"]:
            regenerated_roles[pair_id] = "train"

        for pair_id in validation_df["pair_id"]:
            regenerated_roles[pair_id] = "validation"

        for pair_id in test_df["pair_id"]:
            regenerated_roles[pair_id] = "test"

        saved_roles = dict(
            zip(saved["pair_id"], saved["role"])
        )

        same_ids = set(saved_roles) == set(regenerated_roles)

        same_roles = (
            same_ids
            and all(
                saved_roles[pair_id] == regenerated_roles[pair_id]
                for pair_id in saved_roles
            )
        )

        REPRODUCIBILITY_ROWS.append({
            "family": display_name,
            "outer_fold": outer_fold,
            "same_pair_ids": same_ids,
            "same_roles": same_roles,
            "pass": same_ids and same_roles,
        })

REPRODUCIBILITY_CHECK = pd.DataFrame(
    REPRODUCIBILITY_ROWS
)

assert REPRODUCIBILITY_CHECK["pass"].all(), (
    "At least one regenerated inner split differs from the saved assignment."
)

display(REPRODUCIBILITY_CHECK)

REPRODUCIBILITY_CHECK.to_excel(
    STATS_DIR / "inner_split_reproducibility_check.xlsx",
    index=False,
    sheet_name="Reproducibility",
)

print(
    "Reproducibility check saved to:",
    STATS_DIR / "inner_split_reproducibility_check.xlsx"
)

,family,outer_fold,same_pair_ids,same_roles,pass
0,Enzyme,1,True,True,True
1,Enzyme,2,True,True,True
2,Enzyme,3,True,True,True
3,Enzyme,4,True,True,True
4,Enzyme,5,True,True,True
5,GPCR,1,True,True,True
6,GPCR,2,True,True,True
7,GPCR,3,True,True,True
8,GPCR,4,True,True,True
9,GPCR,5,True,True,True


Reproducibility check saved to: c:\Users\riskf\OneDrive\A-DTI2026\Stats\inner_split_reproducibility_check.xlsx


## 10. Protocol table

The exact split configuration is saved as a statistical/audit table in `Stats/`.

In [12]:
PROTOCOL = pd.DataFrame([
    {
        "parameter": "outer_split_seed",
        "value": OUTER_SPLIT_SEED,
    },
    {
        "parameter": "inner_split_seed",
        "value": INNER_SPLIT_SEED,
    },
    {
        "parameter": "n_outer_folds",
        "value": N_OUTER_FOLDS,
    },
    {
        "parameter": "inner_train_size",
        "value": INNER_TRAIN_SIZE,
    },
    {
        "parameter": "inner_validation_size",
        "value": INNER_VALIDATION_SIZE,
    },
    {
        "parameter": "inner_split_method",
        "value": "sklearn.model_selection.train_test_split",
    },
    {
        "parameter": "inner_stratification",
        "value": "y",
    },
    {
        "parameter": "shuffle",
        "value": True,
    },
])

display(PROTOCOL)

PROTOCOL.to_excel(
    STATS_DIR / "inner_split_protocol.xlsx",
    index=False,
    sheet_name="Protocol",
)

,parameter,value
0,outer_split_seed,13571113
1,inner_split_seed,555487
2,n_outer_folds,5
3,inner_train_size,0.8
4,inner_validation_size,0.2
5,inner_split_method,sklearn.model_selection.train_test_split
6,inner_stratification,y
7,shuffle,True


## 11. Consolidated Notebook 3 statistics workbook



In [13]:
consolidated_file = (
    STATS_DIR / "inner_split_statistics.xlsx"
)

with pd.ExcelWriter(
    consolidated_file,
    engine="openpyxl",
) as writer:

    ROLE_STATS.to_excel(
        writer,
        sheet_name="Role Statistics",
        index=False,
    )

    FOLD_SUMMARY.to_excel(
        writer,
        sheet_name="Fold Summary",
        index=False,
    )

    OVERLAP_CHECK.to_excel(
        writer,
        sheet_name="Overlap Check",
        index=False,
    )

    INNER_RATE_CHECK.to_excel(
        writer,
        sheet_name="Rate Check",
        index=False,
    )

    REPRODUCIBILITY_CHECK.to_excel(
        writer,
        sheet_name="Reproducibility",
        index=False,
    )

    PROTOCOL.to_excel(
        writer,
        sheet_name="Protocol",
        index=False,
    )

print(
    "Consolidated statistics saved to:",
    consolidated_file
)

Consolidated statistics saved to: c:\Users\riskf\OneDrive\A-DTI2026\Stats\inner_split_statistics.xlsx


## 12. Save split manifest



In [14]:
manifest = {
    "step": "06 - fixed inner 80/20 train-validation splits",
    "outer_split_seed": OUTER_SPLIT_SEED,
    "inner_split_seed": INNER_SPLIT_SEED,
    "n_outer_folds": N_OUTER_FOLDS,
    "inner_train_size": INNER_TRAIN_SIZE,
    "inner_validation_size": INNER_VALIDATION_SIZE,
    "stratify_on": "y",
    "families": list(FAMILIES.keys()),
    "assignment_files": {
        family: [
            str(
                (
                    SPLIT_DIR
                    / family
                    / f"outer_fold_{fold}_roles.csv"
                ).relative_to(PROJECT_ROOT)
            )
            for fold in range(1, N_OUTER_FOLDS + 1)
        ]
        for family in FAMILIES
    },
}

manifest_file = (
    SPLIT_DIR / "inner_split_manifest.json"
)

with open(
    manifest_file,
    "w",
    encoding="utf-8",
) as f:
    json.dump(manifest, f, indent=2)

print("Manifest saved to:", manifest_file)

Manifest saved to: c:\Users\riskf\OneDrive\A-DTI2026\data\split\inner_split_manifest.json


## 13. Final integrity check


In [15]:
required_stats_files = [
    "inner_split_role_statistics.xlsx",
    "inner_split_fold_summary.xlsx",
    "inner_split_overlap_check.xlsx",
    "inner_split_rate_check.xlsx",
    "inner_split_reproducibility_check.xlsx",
    "inner_split_protocol.xlsx",
    "inner_split_statistics.xlsx",
]

for family, display_name in FAMILIES.items():

    for outer_fold in range(1, N_OUTER_FOLDS + 1):

        split_file = (
            SPLIT_DIR
            / family
            / f"outer_fold_{outer_fold}_roles.csv"
        )

        assert split_file.exists(), (
            f"{display_name}, fold {outer_fold}: "
            "missing role-assignment file."
        )

        df = pd.read_csv(split_file)

        assert set(df["role"].unique()) == {
            "train",
            "validation",
            "test",
        }

        assert df["pair_id"].is_unique

    print(f"PASS - {display_name}")

assert OVERLAP_CHECK["pass"].all()
assert REPRODUCIBILITY_CHECK["pass"].all()

for filename in required_stats_files:
    assert (STATS_DIR / filename).exists(), (
        f"Missing Stats file: {filename}"
    )

assert (
    SPLIT_DIR / "inner_split_manifest.json"
).exists()

print("\nNotebook 3 completed successfully.")
print(
    "Next: Step 07 - fit matrix factorization "
    "with validation/test labels masked."
)

PASS - Enzyme
PASS - GPCR
PASS - Ion Channel
PASS - Nuclear Receptor

Notebook 3 completed successfully.
Next: Step 07 - fit matrix factorization with validation/test labels masked.
